# Phase 2: Data Cleaning & Master Dataset

This notebook pulls all three raw tables from MySQL, cleans and merges them into a single master dataset, and writes it back to MySQL.

**Steps:**
1. Load `game_prices`, `cpi_data`, and `taketwo_financials` from MySQL
2. Inflation-adjust all prices to 2025 dollar terms using CPI data
3. Merge datasets by year into one master DataFrame
4. Run data quality checks
5. Write the master dataset back to MySQL

**Why inflation-adjust?**
A game priced at $59.99 in 2007 is not the same as $59.99 in 2025. Adjusting all prices to a common base year lets the regression model compare real purchasing power across time rather than nominal sticker prices. Without this step the model would see no meaningful price trend across 16 years.

**Output:** Populated `master_dataset` table in MySQL

## 1. Imports and Database Connection

In [1]:
import pandas as pd
import numpy as np
import mysql.connector
import os
import sys
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine

sys.path.append(os.path.abspath('..'))
load_dotenv(dotenv_path='../.env')

print('Imports successful')

Imports successful


In [2]:
def get_connection():
    """Raw MySQL connection for write operations."""
    return mysql.connector.connect(
        host=os.getenv('DB_HOST'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD'),
        database=os.getenv('DB_NAME')
    )

def get_engine():
    """SQLAlchemy engine for pandas read operations."""
    password = quote_plus(os.getenv('DB_PASSWORD'))
    return create_engine(
        f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{password}"
        f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
    )

try:
    conn = get_connection()
    print(f"Connected to: {os.getenv('DB_NAME')}")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")

Connected to: gtavi_economy_map


## 2. Load Raw Tables from MySQL

In [3]:
engine = get_engine()

df_prices = pd.read_sql("SELECT * FROM game_prices ORDER BY release_year", engine)
df_cpi = pd.read_sql("SELECT year, annual_cpi FROM cpi_data ORDER BY year", engine)
df_financials = pd.read_sql("""
    SELECT fiscal_year, total_revenue_usd_millions,
           gross_margin_pct, net_income_usd_millions
    FROM taketwo_financials
    ORDER BY fiscal_year
""", engine)

print(f"game_prices:       {len(df_prices)} records")
print(f"cpi_data:          {len(df_cpi)} records")
print(f"taketwo_financials:{len(df_financials)} records")

game_prices:       42 records
cpi_data:          26 records
taketwo_financials:14 records


## 3. Preview Raw Data

In [4]:
print("Game prices sample:")
df_prices.head(10)

Game prices sample:


,id,game_title,publisher,release_year,platform,platform_generation,base_price_usd,premium_price_usd,had_premium_edition,created_at
0,7,Call of Duty 4: Modern Warfare,Activision,2007,PS3/Xbox 360,1,59.99,NaN,0,2026-04-29 12:31:09
1,1,Grand Theft Auto IV,Rockstar Games,2008,PS3/Xbox 360,1,59.99,NaN,0,2026-04-29 12:31:09
2,8,Call of Duty: Modern Warfare 2,Activision,2009,PS3/Xbox 360,1,59.99,NaN,0,2026-04-29 12:31:09
3,2,Red Dead Redemption,Rockstar Games,2010,PS3/Xbox 360,1,59.99,NaN,0,2026-04-29 12:31:09
4,9,Call of Duty: Black Ops,Activision,2010,PS3/Xbox 360,1,59.99,NaN,0,2026-04-29 12:31:09
5,22,Battlefield 3,EA,2011,PS3/Xbox 360,1,59.99,NaN,0,2026-04-29 12:31:09
6,10,Call of Duty: Modern Warfare 3,Activision,2011,PS3/Xbox 360,1,59.99,NaN,0,2026-04-29 12:31:09
7,11,Call of Duty: Black Ops II,Activision,2012,PS3/Xbox 360,1,59.99,NaN,0,2026-04-29 12:31:09
8,3,Grand Theft Auto V,Rockstar Games,2013,PS3/Xbox 360,1,59.99,79.99,1,2026-04-29 12:31:09
9,12,Call of Duty: Ghosts,Activision,2013,PS3/Xbox 360,1,59.99,79.99,1,2026-04-29 12:31:09


In [5]:
print("CPI data sample:")
df_cpi.tail(10)

CPI data sample:


,year,annual_cpi
16,2016,240.007
17,2017,245.120
18,2018,251.107
19,2019,255.657
20,2020,258.811
21,2021,270.970
22,2022,292.655
23,2023,304.702
24,2024,313.689
25,2025,321.943


In [6]:
print("Take-Two financials:")
df_financials

Take-Two financials:


,fiscal_year,total_revenue_usd_millions,gross_margin_pct,net_income_usd_millions
0,2012,826.0,35.96,-109.0
1,2013,1214.0,41.10,-29.0
2,2014,2351.0,39.81,321.0
3,2015,1083.0,26.59,-279.0
4,2016,1414.0,42.43,-8.0
5,2017,1780.0,42.53,67.0
6,2018,1793.0,49.92,174.0
7,2019,2668.0,42.92,334.0
8,2020,3089.0,50.08,404.0
9,2021,3373.0,54.49,589.0


## 4. Inflation Adjustment

Convert all nominal prices to 2025 dollar terms.

**Formula:** `real_price = nominal_price × (CPI_2025 / CPI_release_year)`

**Example:** COD4 launched at $59.99 in 2007. CPI in 2007 was ~207. CPI in 2025 is ~322.
Real price = $59.99 × (322 / 207) = ~$93.30 in today's money.

This means Call of Duty 4 was actually a more expensive game in real terms than any title released at $69.99 in 2023 — a genuine and counterintuitive insight your dashboard will visualise.

In [7]:
# Get 2025 as the base year CPI
cpi_2025 = df_cpi.loc[df_cpi['year'] == 2025, 'annual_cpi'].values[0]
print(f"Base year CPI (2025): {cpi_2025}")

# Merge prices with CPI on release year
df = df_prices.merge(
    df_cpi.rename(columns={'year': 'release_year', 'annual_cpi': 'annual_cpi'}),
    on='release_year',
    how='left'
)

# Check for any years missing CPI data
missing = df[df['annual_cpi'].isna()]['release_year'].unique()
if len(missing) > 0:
    print(f"Warning: Missing CPI data for years: {missing}")
else:
    print("CPI data found for all release years")

# Calculate inflation multiplier and real prices
df['cpi_2025'] = cpi_2025
df['inflation_multiplier'] = (cpi_2025 / df['annual_cpi']).round(4)
df['base_price_real'] = (df['base_price_usd'] * df['inflation_multiplier']).round(2)
df['premium_price_real'] = (df['premium_price_usd'] * df['inflation_multiplier']).round(2)

print(f"\nInflation adjustment complete")
df[['game_title', 'release_year', 'base_price_usd', 'inflation_multiplier', 'base_price_real']].head(10)

Base year CPI (2025): 321.943
CPI data found for all release years

Inflation adjustment complete


,game_title,release_year,base_price_usd,inflation_multiplier,base_price_real
0,Call of Duty 4: Modern Warfare,2007,59.99,1.5527,93.15
1,Grand Theft Auto IV,2008,59.99,1.4953,89.70
2,Call of Duty: Modern Warfare 2,2009,59.99,1.5006,90.02
3,Red Dead Redemption,2010,59.99,1.4764,88.57
4,Call of Duty: Black Ops,2010,59.99,1.4764,88.57
5,Battlefield 3,2011,59.99,1.4312,85.86
6,Call of Duty: Modern Warfare 3,2011,59.99,1.4312,85.86
7,Call of Duty: Black Ops II,2012,59.99,1.4022,84.12
8,Grand Theft Auto V,2013,59.99,1.3820,82.91
9,Call of Duty: Ghosts,2013,59.99,1.3820,82.91


## 5. Key Insight — Real Price Trend

Before merging, let's look at what the inflation adjustment reveals. This is one of the core insights of the project.

In [8]:
# Show real vs nominal prices sorted by year
# This reveals the counterintuitive truth: older games were more expensive in real terms
insight = df[['game_title', 'publisher', 'release_year', 'base_price_usd', 'base_price_real', 'inflation_multiplier']].copy()
insight = insight.sort_values('release_year')
insight.columns = ['Game', 'Publisher', 'Year', 'Nominal Price', 'Real Price (2025 $)', 'Inflation Multiplier']

print("Real vs Nominal Prices — what games actually cost in today's money:")
print()
insight.to_string(index=False)

Real vs Nominal Prices — what games actually cost in today's money:



"                                     Game      Publisher  Year  Nominal Price  Real Price (2025 $)  Inflation Multiplier\n           Call of Duty 4: Modern Warfare     Activision  2007          59.99                93.15                1.5527\n                      Grand Theft Auto IV Rockstar Games  2008          59.99                89.70                1.4953\n           Call of Duty: Modern Warfare 2     Activision  2009          59.99                90.02                1.5006\n                      Red Dead Redemption Rockstar Games  2010          59.99                88.57                1.4764\n                  Call of Duty: Black Ops     Activision  2010          59.99                88.57                1.4764\n                            Battlefield 3             EA  2011          59.99                85.86                1.4312\n           Call of Duty: Modern Warfare 3     Activision  2011          59.99                85.86                1.4312\n               Call of 

In [9]:
# Average real price by year — shows the true price trend
avg_by_year = df.groupby('release_year')['base_price_real'].mean().round(2)
print("Average real base price by year (2025 dollars):")
print(avg_by_year.to_string())
print(f"\nOverall average real price: ${df['base_price_real'].mean():.2f}")
print(f"Overall average nominal price: ${df['base_price_usd'].mean():.2f}")

Average real base price by year (2025 dollars):
release_year
2007    93.15
2008    89.70
2009    90.02
2010    88.57
2011    85.86
2012    84.12
2013    82.91
2014    81.58
2015    81.48
2016    80.47
2017    78.79
2018    76.91
2019    75.55
2020    82.91
2021    83.16
2022    74.25
2023    73.95

Overall average real price: $80.49
Overall average nominal price: $62.61


## 6. Merge with Take-Two Financials

We add Take-Two financial context to the dataset. This will be used for dashboard visualisation — showing how Take-Two's revenue growth aligns with the industry pricing shift — rather than as a model feature.

Note: Take-Two's fiscal year ends March 31, so their reported year lags calendar year by one quarter. We match on calendar year directly and note this offset in the README.

In [10]:
df_master = df.merge(
    df_financials.rename(columns={'fiscal_year': 'release_year'}),
    on='release_year',
    how='left'
)

print(f"Master dataset shape: {df_master.shape}")
print(f"\nColumns: {list(df_master.columns)}")

# Check how many rows have Take-Two financial data
with_financials = df_master['total_revenue_usd_millions'].notna().sum()
print(f"\nRows with Take-Two financial data: {with_financials} of {len(df_master)}")
print("Note: Only Rockstar titles will have matching Take-Two financials by year")

Master dataset shape: (42, 18)

Columns: ['id', 'game_title', 'publisher', 'release_year', 'platform', 'platform_generation', 'base_price_usd', 'premium_price_usd', 'had_premium_edition', 'created_at', 'annual_cpi', 'cpi_2025', 'inflation_multiplier', 'base_price_real', 'premium_price_real', 'total_revenue_usd_millions', 'gross_margin_pct', 'net_income_usd_millions']

Rows with Take-Two financial data: 35 of 42
Note: Only Rockstar titles will have matching Take-Two financials by year


## 7. Data Quality Checks

Before writing to MySQL, run systematic checks to catch any issues.

In [11]:
print("DATA QUALITY REPORT")
print()

# Shape
print(f"Total records: {len(df_master)}")
print(f"Total columns: {len(df_master.columns)}")
print()

# Null counts for key columns
key_cols = ['game_title', 'publisher', 'release_year', 'platform_generation',
            'base_price_usd', 'base_price_real', 'inflation_multiplier']
print("Null counts (key columns):")
print(df_master[key_cols].isnull().sum().to_string())
print()

# Price range sanity
print(f"Base price range: ${df_master['base_price_usd'].min()} to ${df_master['base_price_usd'].max()}")
print(f"Real price range: ${df_master['base_price_real'].min()} to ${df_master['base_price_real'].max()}")
print()

# Year range
print(f"Year range: {df_master['release_year'].min()} to {df_master['release_year'].max()}")
print()

# Publisher distribution
print("Records per publisher:")
print(df_master['publisher'].value_counts().to_string())
print()

# Platform generation distribution
print("Records per platform generation:")
print(df_master['platform_generation'].value_counts().sort_index().to_string())
print()

# Duplicate check
dupes = df_master[df_master.duplicated(subset=['game_title', 'release_year'], keep=False)]
print(f"Duplicates: {len(dupes)} records")

DATA QUALITY REPORT

Total records: 42
Total columns: 18

Null counts (key columns):
game_title              0
publisher               0
release_year            0
platform_generation     0
base_price_usd          0
base_price_real         0
inflation_multiplier    0

Base price range: $59.99 to $69.99
Real price range: $65.99 to $93.15

Year range: 2007 to 2023

Records per publisher:
publisher
Activision        15
EA                 9
Sony               8
Rockstar Games     6
Nintendo           4

Records per platform generation:
platform_generation
1    12
2    18
3    12

Duplicates: 0 records


## 8. Write Master Dataset to MySQL

In [13]:
def write_master_dataset(df):
    """
    Write the cleaned master dataset to MySQL.
    Uses try/finally to guarantee connections close even if an error occurs.
    This prevents the table metadata lock issue caused by orphaned connections.
    """
    conn = get_connection()
    cursor = conn.cursor()

    try:
        cursor.execute("DELETE FROM master_dataset")

        insert_query = """
            INSERT INTO master_dataset (
                game_title, publisher, release_year, platform,
                platform_generation, had_premium_edition,
                base_price_nominal, premium_price_nominal,
                base_price_real, premium_price_real,
                annual_cpi, cpi_2025, inflation_multiplier,
                revenue_usd_millions, gross_margin_pct, net_income_usd_millions
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        def val(v):
            """Convert NaN to None for MySQL compatibility."""
            return None if pd.isna(v) else v

        records = []
        for _, row in df.iterrows():
            records.append((
                row['game_title'],
                row['publisher'],
                int(row['release_year']),
                row['platform'],
                int(row['platform_generation']),
                int(row['had_premium_edition']),
                val(row['base_price_usd']),
                val(row['premium_price_usd']),
                val(row['base_price_real']),
                val(row['premium_price_real']),
                val(row['annual_cpi']),
                val(row['cpi_2025']),
                val(row['inflation_multiplier']),
                val(row.get('total_revenue_usd_millions')),
                val(row.get('gross_margin_pct')),
                val(row.get('net_income_usd_millions')),
            ))

        cursor.executemany(insert_query, records)
        conn.commit()
        print(f"Written {cursor.rowcount} records to master_dataset")

    finally:
        cursor.close()
        conn.close()

write_master_dataset(df_master)

Written 42 records to master_dataset


## 9. Verify from MySQL

Read back from MySQL to confirm the round-trip was clean.

In [14]:
df_verify = pd.read_sql("""
    SELECT
        release_year,
        publisher,
        game_title,
        platform_generation,
        base_price_nominal,
        base_price_real,
        inflation_multiplier
    FROM master_dataset
    ORDER BY release_year, publisher
""", engine)

print(f"Records in master_dataset: {len(df_verify)}")
df_verify

Records in master_dataset: 42


,release_year,publisher,game_title,platform_generation,base_price_nominal,base_price_real,inflation_multiplier
0,2007,Activision,Call of Duty 4: Modern Warfare,1,59.99,93.15,1.5527
1,2008,Rockstar Games,Grand Theft Auto IV,1,59.99,89.70,1.4953
2,2009,Activision,Call of Duty: Modern Warfare 2,1,59.99,90.02,1.5006
3,2010,Activision,Call of Duty: Black Ops,1,59.99,88.57,1.4764
4,2010,Rockstar Games,Red Dead Redemption,1,59.99,88.57,1.4764
5,2011,Activision,Call of Duty: Modern Warfare 3,1,59.99,85.86,1.4312
6,2011,EA,Battlefield 3,1,59.99,85.86,1.4312
7,2012,Activision,Call of Duty: Black Ops II,1,59.99,84.12,1.4022
8,2013,Activision,Call of Duty: Ghosts,1,59.99,82.91,1.3820
9,2013,EA,FIFA 14,1,59.99,82.91,1.3820
